In [ ]:
# ========================================
# 第1部分:项目介绍
# ========================================

"""
# 项目名称
ReportWriter —— 智能材料撰写 Agent

## 项目简介
ReportWriter 是一个基于 HelloAgents v1.0.0 框架的多类型材料撰写智能体，
能够撰写工作总结、汇报、KPI 计划等不同类型的材料，并支持扩展更多类型。
核心特色：
1. 多类型可扩展：新增一类材料 ≈ 新增一个 Spec 配置，零侵入核心代码
2. 参考材料导入：用户可导入写好的范文/历史材料，Agent 撰写时参考其风格与口径
3. 多范式协作：PlanAndSolve 规划 + ReAct 撰写 + Reflection 评审
4. 双格式导出：Markdown + DOCX

## 架构概览
PlannerAgent(PlanAndSolve) → DraftingAgent(ReAct+RAG工具) → ReviewAgent(Reflection) → Exporter(MD+DOCX)
                                  ↑
                          MaterialManager(RAG 参考材料)

## 作者信息
- 姓名: MadDoggy
- GitHub: @EricPeng1027
- 日期: 2026-08-13
"""

In [ ]:
# ========================================
# 第2部分:环境配置
# ========================================

# 安装依赖（首次运行取消注释）
# !pip install -q hello-agents[all] python-docx python-dotenv pydantic-settings \
#     qdrant-client openai markitdown

import sys
from pathlib import Path

# 将 src 加入模块搜索路径（notebook 直接运行用）
SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dotenv import load_dotenv
load_dotenv(override=True)

# 导入项目模块
from src.config import get_settings
from src.registry import get_registry

settings = get_settings()
print("✅ 环境配置完成")
print(f"   LLM 模型: {settings.llm_model_id}")
print(f"   参考材料目录: {settings.data_dir}")
print(f"   输出目录: {settings.output_dir}")
print(f"   RAG 模式: Qdrant={settings.qdrant_url[:30]}... | Embed={settings.embed_model_type}")

In [ ]:
# ========================================
# 第3部分:工具与材料层定义
# ========================================

# ReportWriter 的工具与材料层封装在 src/ 中：
# - MaterialManager: 导入用户参考材料，提供主动注入与被动召回两种参考方式
# - recall_material 工具: ReAct 写作中按需检索参考材料
#
# RAG 后端基于 Qdrant + OpenAI 兼容 Embedding 自建（见 src/materials/rag_backend.py）

from src.materials.manager import MaterialManager
from src.tools.recall_material import build_recall_material_tool

# 演示材料导入与检索
material_mgr = MaterialManager(mode="rag", namespace="reportwriter_demo")
print(f"   RAG 后端就绪: {material_mgr.ready}")
if not material_mgr.ready:
    print(f"   原因: {material_mgr.error}")

# 导入 data/ 下的参考材料（若目录为空或后端未就绪则跳过）
if material_mgr.ready:
    ingest_result = material_mgr.ingest(settings.data_dir)
    print(f"✅ 参考材料层就绪: 导入 {ingest_result['success']}/{ingest_result['total']} 个文件")
else:
    print("⚠️ RAG 后端未就绪，撰写时将跳过参考材料（仍可生成，仅无参考）")

print(f"\n可用材料类型:")
print(get_registry().describe())

In [ ]:
# ========================================
# 第4部分:智能体构建
# ========================================

# Orchestrator 内部自动构建三 Agent 并组装流水线：
#   PlannerAgent  → PlanAndSolveAgent（章节大纲）
#   DraftingAgent → ReActAgent + RecallMaterialTool（逐节撰写）
#   ReviewAgent   → ReflectionAgent（自评修订）

from src.orchestrator import ReportWriterOrchestrator

orchestrator = ReportWriterOrchestrator(material_namespace="reportwriter_demo")

print("\n✅ 智能体流水线构建完成")
print(f"   可用材料类型: {orchestrator.list_types()}")

In [5]:
# ========================================
# 第5部分:功能演示
# ========================================

# 示例1:撰写工作总结
print("=== 示例1:撰写工作总结 ===")
draft = orchestrator.write(
    type_id="work_summary",
    topic="2026年Q2 研发部工作总结",
    materials_dir=settings.data_dir,
)

print("\n---------- 撰写结果预览 ----------")
print(draft.full_markdown()[:1500])
print("...\n(完整内容见 outputs/ 目录下的 .md / .docx 文件)")

--- 计划已生成 ---
-> 执行步骤 1/8: 步骤1: 分析主题与目标读者，明确材料定位——本材料为2026年Q2研发部工作总结，面向公司管理层及部门成员，定位为期中复盘材料，需兼顾事实陈述与价值提炼，语言专业、简洁、得体
-> 执行步骤 2/8: 步骤2: 收集并梳理原始素材——汇总Q2各项目进度数据、版本发布记录、关键指标(交付量、质量、效率等)、团队建设情况及问题清单，按项目/模块归类，筛选可用的事实与数据
-> 执行步骤 3/8: 步骤3: 梳理整体结构与逻辑主线——确立'总体概述→重点亮点→具体工作→问题不足→下期计划'的递进结构，逻辑主线为'做了什么—做得怎样—有何不足—下一步怎么办'
-> 执行步骤 4/8: 步骤4: 规划'总体概述'章节(约300字)——交代Q2工作背景与季度目标(如OKR对齐情况)、整体完成进度(用完成率等数据概括全局)、部门核心定位与本季度工作基调
-> 执行步骤 5/8: 步骤5: 规划'重点工作与亮点'章节(约600字)——筛选3-4项核心成果与关键突破(如重要版本上线、技术攻关、降本增效)，每项均以量化数据与事实支撑，突出业务价值与部门贡献
-> 执行步骤 6/8: 步骤6: 规划'具体工作内容'章节(约800字)——按项目/模块/职责分项展开，每项说明'做了什么、怎么做、达成了什么'，与亮点章节形成详略互补，避免内容重复
-> 执行步骤 7/8: 步骤7: 规划'问题与不足'与'下期计划'章节(各约300字)——问题部分客观陈述2-3项不足及已采取的应对措施；计划部分与问题呼应，提出Q3目标、重点工作与改进方向，形成'复盘—改进'闭环
-> 执行步骤 8/8: 步骤8: 组装完整章节大纲并校验——整合各章节要点、字数分配与数据要求，检查与章节骨架(key与标题)的一致性及各章节间衔接逻辑，输出最终大纲
▸ 规划完成: 2026年Q2研发部工作总结，5 章

▸ 第三步：逐节撰写（ReAct + 参考材料）

──────────────────────────────────────────────────────────────────────
▸ 第 1/5 章: 总体概述
────────────────────────────────────────────────────────────────────

In [6]:
# ========================================
# 第6部分:性能评估（可选）
# ========================================

# 统计本次撰写的字数、耗时与参考材料命中情况
print("=== 性能评估 ===")
print(f"材料类型: {draft.type_id}")
print(f"总字数: {draft.total_words()}")
print(f"耗时: {draft.meta.get('duration', 0):.1f} 秒")
print(f"参考材料命中片段数: {draft.meta.get('material_hits', 0)}")
print(f"使用范式: {draft.meta.get('paradigm')}")
print("\n各章节字数:")
for sec in draft.sections:
    reviewed = sec.metadata.get('reviewed', False)
    print(f"  - {sec.title}: {sec.word_count} 字 {'(已评审)' if reviewed else '(未评审)'}")

=== 性能评估 ===
材料类型: work_summary
总字数: 2379
耗时: 2404.9 秒
参考材料命中片段数: 2
使用范式: plan_solve

各章节字数:
  - 总体概述: 254 字 (已评审)
  - 重点工作与亮点: 489 字 (已评审)
  - 具体工作内容: 901 字 (已评审)
  - 问题与不足: 276 字 (已评审)
  - 下期计划: 459 字 (已评审)


In [7]:
# ========================================
# 第7部分:总结与展望
# ========================================

"""
## 项目总结

### 实现的功能
- 基于三种 HelloAgents 范式（PlanAndSolve / ReAct / Reflection）的多 Agent 材料撰写流水线
- 工作总结类型（P1）端到端跑通，输出 Markdown + DOCX
- RAG 参考材料层：导入范文/历史材料，撰写时主动注入 + 被动召回双模式参考
- 可扩展的类型注册表：新增材料类型只需新增 Spec 配置

### 遇到的挑战
- ReAct 输出 JSON 不稳定 → JSONExtractor 多策略提取 + SimpleAgent 回退兜底
- PlanAndSolve 大纲可能与章节骨架不对齐 → normalize 补齐缺失章节
- RAG 依赖 Qdrant + Embedding 外部服务 → 配置集中在 .env，模式可切换

### 未来改进方向
- P2: 补充『汇报』『KPI 计划』等材料类型 Spec，验证零侵入扩展
- 增加材料类型的『本地后端』兜底，无 Qdrant 时也能用关键词匹配
- 接入更多工具（数据表读取、图表生成）支撑数据型材料
- 多轮交互式撰写：支持用户对大纲/章节给出反馈后增量修订
"""

'\n## 项目总结\n\n### 实现的功能\n- 基于三种 HelloAgents 范式（PlanAndSolve / ReAct / Reflection）的多 Agent 材料撰写流水线\n- 工作总结类型（P1）端到端跑通，输出 Markdown + DOCX\n- RAG 参考材料层：导入范文/历史材料，撰写时主动注入 + 被动召回双模式参考\n- 可扩展的类型注册表：新增材料类型只需新增 Spec 配置\n\n### 遇到的挑战\n- ReAct 输出 JSON 不稳定 → JSONExtractor 多策略提取 + SimpleAgent 回退兜底\n- PlanAndSolve 大纲可能与章节骨架不对齐 → normalize 补齐缺失章节\n- RAG 依赖 Qdrant + Embedding 外部服务 → 配置集中在 .env，模式可切换\n\n### 未来改进方向\n- P2: 补充『汇报』『KPI 计划』等材料类型 Spec，验证零侵入扩展\n- 增加材料类型的『本地后端』兜底，无 Qdrant 时也能用关键词匹配\n- 接入更多工具（数据表读取、图表生成）支撑数据型材料\n- 多轮交互式撰写：支持用户对大纲/章节给出反馈后增量修订\n'